# 특허 기술이전 수요예측 — Colab GPU 실행

`run_ipm_experiment.py`를 **무료 GPU(CUDA)** 에서 full 모드로 실행합니다. 맥북 뚜껑/잠자기 걱정 없이 클라우드에서 돌아갑니다.

**순서**: ① GPU 켜기 → ② 코드 받기 → ③ 라이브러리 → ④ 데이터(구글드라이브) → ⑤ 실행 → ⑥ 결과 저장

> ⚠️ 먼저 상단 메뉴 **런타임 ▸ 런타임 유형 변경 ▸ 하드웨어 가속기: T4 GPU** 로 설정하세요.
> 무료 Colab은 세션 ~12시간/유휴 끊김이 있으니 탭을 켜 두세요. 10-seed가 길면 ⑤의 `--seeds 3`부터 권장.


## ① GPU 확인


In [ ]:
!nvidia-smi


## ② 코드 받기 (GitHub)


In [ ]:
!git clone https://github.com/joshlee614/Patent-Citation-Graph-Based-Technology-Transfer-Demand-Prediction-.git repo
%cd repo
!git log --oneline -3


## ③ 라이브러리 설치
torch는 Colab에 이미 설치돼 있습니다. PyG와 statsmodels만 추가합니다.


In [ ]:
!pip install -q torch_geometric statsmodels
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())


## ④ 데이터 연결 (Google Drive)

GitHub에는 데이터가 없습니다. **구글 드라이브에 아래 4개 파일만** 올리세요:

- `patents.csv`  ·  `transfers.csv`  ·  `citings.csv`  ·  `patent_embeddings.pt`

❌ `claims.csv` / `legalStatus.csv` 등 큰 파일은 **불필요**합니다(스크립트가 안 씀).

예: 드라이브에 `kipris_data` 폴더를 만들고 그 안에 4개 파일을 넣으면 경로가 `/content/drive/MyDrive/kipris_data` 입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, torch
DATA_DIR = '/content/drive/MyDrive/kipris_data'   # ← 본인 드라이브 경로로 수정
EMB = os.path.join(DATA_DIR, 'patent_embeddings.pt')
for f in ['patents.csv','transfers.csv','citings.csv','patent_embeddings.pt']:
    p = os.path.join(DATA_DIR, f)
    print(('OK  ' if os.path.exists(p) else 'MISSING '), p)
print('embedding shape:', tuple(torch.load(EMB, map_location='cpu').shape), '(patents.csv 행수와 같아야 함)')


## ⑤ 실행
먼저 `--seeds 3`으로 빠르게 검증/첫 수치를 보고, 문제 없으면 아래 ⑥에서 전체 10-seed를 돌리세요.
`--demand_sample 2000`은 느리고 거의 무의미한 Demand(E19)를 표본으로 제한합니다.


In [ ]:
!python run_ipm_experiment.py --mode full --seeds 3 --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" \
  --demand_sample 2000 --artifact_dir ./ipm_artifacts_s3


## ⑥ 전체 10-seed (최종 결과)


In [ ]:
!python run_ipm_experiment.py --mode full --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" \
  --demand_sample 2000 --artifact_dir ./ipm_artifacts


## 결과 확인 + 드라이브에 저장


In [ ]:
print(open('ipm_artifacts/run_ipm_results.md').read())


In [ ]:
!cp -r ipm_artifacts /content/drive/MyDrive/kipris_data/ipm_results
print('결과 표/그림을 드라이브 kipris_data/ipm_results 에 저장했습니다.')
